#  Climate Change and Extreme Weather Events

**Research Question:** How have the frequency and economic damage of extreme weather events changed
over the past 50 years, and which world regions show the steepest acceleration?

In [52]:
import pandas as pd
import missingno as msno
import pycountry_convert as pc

### STEP 1: Data Loading, Audit, and Preprocessing

Temos 3 dataset diferentes que vamos limpar e organizar para transformar em um único conjunto de dados para depois fazermos as análises e gráficos.

In [87]:
line = 60 * '-'
ds1 = 'economic-damages-from-disasters-as-a-share-of-gdp'
ds2 = 'per-capita-greenhouse-gas-emissions'
ds3 = 'temperature-anomaly'

df1 = pd.read_csv(f'../data/{ds1}/{ds1}.csv')
df2 = pd.read_csv(f'../data/{ds2}/{ds2}.csv')
df3 = pd.read_csv(f'../data/{ds3}/{ds3}.csv')

n_rows_1, n_cols_1 = df1.shape
n_rows_2, n_cols_2 = df2.shape
n_rows_3, n_cols_3 = df3.shape

print(f"O dataset {ds1} tem {n_rows_1} linhas e {n_cols_1} colunas.")
print(f"O dataset {ds2} tem {n_rows_2} linhas e {n_cols_2} colunas.")
print(f"O dataset {ds3} tem {n_rows_3} linhas e {n_cols_3} colunas.")

O dataset economic-damages-from-disasters-as-a-share-of-gdp tem 5615 linhas e 13 colunas.
O dataset per-capita-greenhouse-gas-emissions tem 36179 linhas e 4 colunas.
O dataset temperature-anomaly tem 528 linhas e 6 colunas.


### Análise do dataset **Economic damages from disasters as a share of GDP (%)**

O primeiro dataset são os danos económicos (em % do GDP relativa a cada país) que os desastres causaram entre 1975 e 2024 (50 anos).

Os dados contém 13 colunas e 5.615 linhas. A coluna Entity apresenta o nome de 199 países mais 6 agrupamentos (o mundo todo e mais grupos classficados de acordo com a receita do país). A coluna Code é o respectivo código da Entity. As restantes colunas, além do ano, são os valores de dano por tipo de desastre. Como nem todos os desastres estão relacionados com as condições climáticas, resolvemos eliminar as colunas [Earthquakes, Volcanoes, Glacial lake outbursts, Mass movement (dry), Mass movement (wet)], ficando apenas com [Droughts, Floods, Storms, Extreme temperatures, Wildfires].

Em seguida tivemos que utilizar a biblioteca pycountry_convert para identificar a qual continente pertencia cada país, visto que esta informação não estava no dataset e os agrupamentos eram pelo tamanho da economia de cada país. Ajustamos as colunas e utilizamos a mediana e não a média para agrupar os dados por continente/região e ano pois evita que um único país com um desastre extremo distorça a realidade do continente inteiro.

Há valores vazinhos em algumas colunas, normalmente indicando a inexistência ou indisponibilidade de dados em um determinado ano para um tipo de desastre. E não há valores duplicados.

O dano total é definido como o valor de todas as perdas econômicas direta ou indiretamente causadas pelo desastre, sem ajuste pela inflação.

Citação:

> EM-DAT, CRED / UCLouvain (2025); National statistical organizations and central banks, OECD national accounts, and World Bank staff estimates (2025) – with major processing by Our World in Data

> EM-DAT, CRED / UCLouvain (2025); National statistical organizations and central banks, OECD national accounts, and World Bank staff estimates (2025) – with major processing by Our World in Data. “Annual economic damages as a share of GDP from droughts – EM-DAT” [dataset]. EM-DAT, “The International Disasters Database”; National statistical organizations and central banks, OECD national accounts, and World Bank staff estimates, “World Development Indicators 122” [original data].

In [88]:
# Manter apenas as colunas que representam tipos de desastres que podem estar ligados a mudanças climáticas
df1.info()

print('\n' + line + '\n')

# Manter apenas as colunas 0, 1 e 2, 3, 4, 6, 7 e 9
df1_disasters = df1.iloc[:, [0, 1, 2, 3, 4, 6, 7, 9]].copy()

df1_disasters.iloc[:, 3:8].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5615 entries, 0 to 5614
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Entity                  5615 non-null   object 
 1   Code                    5615 non-null   object 
 2   Year                    5615 non-null   int64  
 3   Droughts                1499 non-null   float64
 4   Floods                  3436 non-null   float64
 5   Earthquakes             1104 non-null   float64
 6   Storms                  2399 non-null   float64
 7   Extreme temperatures    886 non-null    float64
 8   Volcanoes               382 non-null    float64
 9   Wildfires               575 non-null    float64
 10  Glacial lake outbursts  13 non-null     float64
 11  Mass movement (dry)     94 non-null     float64
 12  Mass movement (wet)     851 non-null    float64
dtypes: float64(10), int64(1), object(2)
memory usage: 570.4+ KB

--------------------------------

,Droughts,Floods,Storms,Extreme temperatures,Wildfires
count,1499.000000,3436.000000,2399.000000,886.000000,575.000000
mean,0.137449,0.196996,1.499610,0.051535,0.322951
std,0.676004,1.332494,11.741689,0.730762,5.443686
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.005583,0.000000,0.000000
75%,0.019823,0.039862,0.095352,0.000000,0.006269
max,16.941236,56.383923,258.451350,16.274971,127.277640


In [95]:
# Filtrar apenas os dados relativos a regiões (continentes) do planeta
def country_to_continent(country_code):
    try:
        country_code_alpha2 = pc.country_alpha3_to_country_alpha2(country_code)
        continent_code = pc.country_alpha2_to_continent_code(country_code_alpha2)
        continent_name = pc.convert_continent_code_to_continent_name(continent_code)
        return continent_name
    except Exception as e:
        # print(f"Erro ao converter {country_code}: {e}")
        # TLS - Timor-Leste (Asia), SXM - Sint Maarten (North America)
        if country_code == 'TLS':
            return "Asia"
        if country_code == 'SXM':
            return "North America"
        return "Other/Unknown"
    
# Eliminar as linhas que correspondem a regiões globais (OWID) e adicionar uma coluna de região (continente)
df1_regions = df1_disasters[~df1_disasters.Code.str.startswith('OWID')].copy()
df1_regions['Region'] = df1_regions['Code'].apply(country_to_continent)

# Agrupar os dados por região e ano, somando os valores dos desastres para cada região
df1_regions = df1_regions.drop(columns=['Code', 'Entity'])
# df1_regions = df1_regions.groupby(['Region', 'Year']).agg(['mean', 'median', 'max']).reset_index()
df1_regions = df1_regions.groupby(['Region', 'Year']).median().reset_index()

df1_regions.info()
df1_regions.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 385 entries, 0 to 384
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Region                385 non-null    object 
 1   Year                  385 non-null    int64  
 2   Droughts              276 non-null    float64
 3   Floods                355 non-null    float64
 4   Storms                346 non-null    float64
 5   Extreme temperatures  170 non-null    float64
 6   Wildfires             196 non-null    float64
dtypes: float64(5), int64(1), object(1)
memory usage: 21.2+ KB


,Year,Droughts,Floods,Storms,Extreme temperatures,Wildfires
count,385.000000,276.000000,355.000000,346.000000,170.000000,196.000000
mean,1992.381818,0.157548,0.078814,0.226953,0.085670,0.463166
std,18.602536,0.634000,0.380607,0.861318,0.938550,4.662479
min,1960.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1976.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1992.000000,0.000000,0.000514,0.002774,0.000000,0.000000
75%,2008.000000,0.079316,0.022260,0.053524,0.000000,0.014896
max,2024.000000,9.224813,4.779224,7.418616,12.226847,63.638820


In [96]:
missing = df1_regions.isnull().sum()
pct = (missing / len(df1_regions) * 100).round(1)
summary = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': pct
})

if summary['missing_count'].sum() == 0:
    print("Não há valores ausentes no dataset.")
else:
    print("Valores ausentes por coluna:")
    print(summary[summary['missing_count'] > 0])

Valores ausentes por coluna:
                      missing_count  missing_pct
Droughts                        109         28.3
Floods                           30          7.8
Storms                           39         10.1
Extreme temperatures            215         55.8
Wildfires                       189         49.1


In [97]:
duplicates = df1_regions.duplicated().sum()

print(f"Número de linhas duplicadas: {duplicates}")

Número de linhas duplicadas: 0


### Análise do dataset **Per capita greenhouse gas emissions**